<div style="color: #0B39C9; font-size: 24px; font-weight: 700;">

DỰ ÁN: XÂY DỰNG MÔ HÌNH PHÂN LOẠI VÀ DỰ BÁO RỦI RO KHÁCH HÀNG VAY VỐN

</div>

# <span style="color: #E60000;">Notebook 06. Machine Learning</span>

*Huấn luyện, đánh giá và lựa chọn mô hình phân loại rủi ro tín dụng từ bộ đặc trưng `application_features` để bàn giao sang Notebook 07*

---

**Input:** Bảng `application_features` trong PostgreSQL (kết quả của Notebook 05), gồm dữ liệu đã làm sạch và các đặc trưng được lựa chọn cho mô hình.

**Output:** Mô hình được lựa chọn, các artifact tiền xử lý cần thiết và metadata gồm danh sách feature, chỉ số đánh giá cùng ngưỡng dự đoán để bàn giao sang Notebook 07.

**Pipeline:** Business Understanding → Data Understanding → Database Organization → Data Cleaning → EDA & Visualization → Feature Engineering → **Machine Learning** → Prediction Demo


## I. Giới thiệu

### 1. Mục tiêu của Notebook 06

Huấn luyện và so sánh Logistic Regression, Random Forest và XGBoost để chọn mô hình dự đoán rủi ro tín dụng phù hợp.

### 2. Vai trò của Machine Learning trong dự án


Mô hình biến các đặc trưng đã chuẩn bị thành xác suất rủi ro, từ đó hỗ trợ đánh giá hồ sơ vay một cách nhất quán.

### 3. Mạch liên kết: Notebook 05 → Notebook 06 → Notebook 07

Notebook 05 bàn giao bảng `application_features`; Notebook 06 chọn mô hình và lưu thông tin cần thiết để Notebook 07 thực hiện dự đoán.

## II. Đọc dữ liệu

### 1. Kết nối PostgreSQL và đọc bảng `application_features`

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine

In [2]:
env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
if not env_path.exists():
    raise FileNotFoundError("Không tìm thấy file .env chứa cấu hình PostgreSQL.")

load_dotenv(env_path)
db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER", "postgres"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME", "credit_risk_db"),
)
engine = create_engine(db_url)

df = pd.read_sql("SELECT * FROM public.application_features", engine)
assert {"sk_id_curr", "target"}.issubset(df.columns), (
    "Bảng application_features phải có sk_id_curr và target."
)
print(f"Đã đọc application_features: {df.shape[0]:,} dòng × {df.shape[1]:,} cột")
display(df.head())


Đã đọc application_features: 305,181 dòng × 157 cột


,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,bureau_recency_days,previous_credit_to_current,previous_recency_days,installments_payment_ratio,has_installments_late,has_pos_cash_dpd,credit_card_utilization,age_income_interaction,late_debt_interaction,ext_ltv_interaction
0,100002,1,Cash loans,M,N,Y,0.0,202500.0,406597.5,24700.5,...,103.0,0.440374,606.0,1.000000,0,0,NaN,25-34 | Cao,Không ghi nhận trễ | Trung bình,0.970984
1,100003,0,Cash loans,F,N,N,0.0,270000.0,1293502.5,35698.5,...,606.0,0.374326,746.0,1.000000,0,0,NaN,45-54 | Rất cao,Không ghi nhận trễ | Không còn dư nợ,0.584514
2,100004,0,Revolving loans,M,Y,Y,0.0,67500.0,135000.0,6750.0,...,408.0,0.148933,815.0,1.000000,0,0,NaN,45-54 | Rất thấp,Không ghi nhận trễ | Không còn dư nợ,0.402841
3,100006,0,Cash loans,F,N,Y,0.0,135000.0,312682.5,29686.5,...,NaN,0.932881,181.0,1.000000,0,0,0.0,45-54 | Thấp,NaN,0.459122
4,100007,0,Cash loans,M,N,Y,0.0,121500.0,513000.0,21865.5,...,1149.0,0.324832,374.0,0.964285,1,0,NaN,45-54 | Thấp,Từng trễ | Không còn dư nợ,0.545329


**Nhận xét:** `application_features` là đầu vào duy nhất của NB06; các bước sau không đọc lại CSV thô.

### 2. Kiểm tra kích thước và kiểu dữ liệu


In [3]:
input_summary = pd.DataFrame({
    "Chỉ tiêu": ["Số dòng", "Số cột", "Số cột số", "Số cột phân loại", "Tổng giá trị thiếu"],
    "Kết quả": [
        len(df),
        df.shape[1],
        df.select_dtypes(include="number").shape[1],
        df.select_dtypes(exclude="number").shape[1],
        int(df.isna().sum().sum()),
    ],
})

display(input_summary)
display(
    df.dtypes.value_counts()
    .rename_axis("Kiểu dữ liệu")
    .reset_index(name="Số cột")
)


,Chỉ tiêu,Kết quả
0,Số dòng,305181
1,Số cột,157
2,Số cột số,138
3,Số cột phân loại,19
4,Tổng giá trị thiếu,2725936


,Kiểu dữ liệu,Số cột
0,float64,95
1,int64,43
2,str,19


**Nhận xét:** Kết quả này xác định các biến cần xử lý missing, mã hóa hoặc chuẩn hóa ở Mục III.

### 3. Khảo sát biến mục tiêu `target`

In [4]:
target_summary = (
    df["target"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="Số lượng")
)
target_summary["Tỷ lệ (%)"] = (
    target_summary["Số lượng"] / len(df) * 100
).round(2)

assert set(df["target"].dropna().unique()).issubset({0, 1}), (
    "target chỉ được chứa hai lớp 0 và 1."
)
display(target_summary)


,target,Số lượng,Tỷ lệ (%)
0,0,280462,91.9
1,1,24719,8.1


**Nhận xét:** Phân bố `target` là cơ sở để chia train/test có giữ tỷ lệ lớp ở Mục III; Accuracy không được dùng làm chỉ số duy nhất.

## III. Chuẩn bị dữ liệu

### 1. Xác định Feature và Target

In [5]:
ID_COLUMN = "sk_id_curr"
TARGET_COLUMN = "target"

excluded_columns = [ID_COLUMN, TARGET_COLUMN]
X = df.drop(columns=excluded_columns)
y = df[TARGET_COLUMN].copy()

assert ID_COLUMN not in X.columns, "sk_id_curr không được đưa vào mô hình."
assert TARGET_COLUMN not in X.columns, "target không được đưa vào feature."
assert len(X) == len(y), "Feature và target phải có cùng số dòng."

print(f"Số feature ban đầu: {X.shape[1]:,}")
print(f"Số quan sát: {len(y):,}")
display(pd.DataFrame({
    "Vai trò": ["Feature (X)", "Target (y)"],
    "Nội dung": [f"{X.shape[1]:,} cột đầu vào", TARGET_COLUMN],
}))


Số feature ban đầu: 155
Số quan sát: 305,181


,Vai trò,Nội dung
0,Feature (X),155 cột đầu vào
1,Target (y),target


**Nhận xét:** `sk_id_curr` chỉ là mã định danh, còn `target` là nhãn cần dự đoán nên đều không được đưa vào `X`. Các feature còn lại sẽ được rà soát ở các mục tiếp theo.

### 2. Chia tập Train/Test

### 3. Tạo lại Interaction Features sau khi chia Train/Test

### 4. Xử lý giá trị thiếu

### 5. Mã hóa dữ liệu

### 6. Chuẩn hóa dữ liệu khi cần

## IV. Mô hình Logistic Regression

### 1. Huấn luyện mô hình

### 2. Dự đoán và đánh giá mô hình

## V. Mô hình Random Forest

### 1. Huấn luyện mô hình

### 2. Dự đoán và đánh giá mô hình

### 3. Feature Importance

## VI. Mô hình XGBoost

### 1. Huấn luyện mô hình

### 2. Dự đoán và đánh giá mô hình

### 3. Feature Importance

## VII. So sánh các mô hình

### 1. Bảng tổng hợp kết quả

### 2. Nhận xét kết quả đánh giá

### 3. Đánh giá đóng góp của các feature mới

## VIII. Lựa chọn mô hình cuối cùng

### 1. Lý do lựa chọn mô hình

### 2. Chọn ngưỡng dự đoán

### 3. Lưu mô hình và thông tin kèm theo

### 4. Bàn giao sang Notebook 07

## IX. Tổng kết
